# RiskGuard 🛡️
## Phase 3: Comprehensive Data Exploration & Schema Profiling

**Goal:** RiskGuard is a data-driven infrastructure project delay prediction and risk assessment platform. This notebook performs a rigorous exploratory data audit across all 8 provided datasets from `data/raw/` prior to any cleaning or feature engineering.

### Audit Checklist per Dataset:
1. File name, format, and dimensions
2. Column names and raw data types
3. Missing value counts & percentages
4. Duplicate row detection
5. Categorical cardinality and top unique values
6. Geographic keys (State, District, Coordinates)
7. Date & timeline fields
8. Numeric & financial features
9. Candidate primary keys & join keys
10. Candidate target variables & data leakage risks
11. Suspicious, inconsistent, or placeholder values


In [1]:
import os
import re
import pandas as pd
import numpy as np
import pypdf

# Display configuration
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 50)
pd.set_option('display.width', 1000)

RAW_DIR = os.path.join("..", "data", "raw")
print("Raw data root:", os.path.abspath(RAW_DIR))


Raw data root: C:\Users\RABIYA BUSHRA\OneDrive\Attachments\Desktop\SIH\Implementation\RiskGuard\data\raw


---
## 1. PAIMANA Infrastructure Project Dataset (`FlashReport_March_2026.pdf`)
- **Source:** Ministry of Statistics and Programme Implementation (MoSPI) / PAIMANA
- **Purpose:** Primary project-level infrastructure monitoring dataset


In [2]:
paimana_pdf_path = os.path.join(RAW_DIR, "paimana", "FlashReport_March_2026.pdf")
print("PAIMANA PDF Path:", paimana_pdf_path)
print("File size (MB):", round(os.path.getsize(paimana_pdf_path) / (1024 * 1024), 2))

reader = pypdf.PdfReader(paimana_pdf_path)
total_pages = len(reader.pages)
print(f"Total pages in report: {total_pages}")

# Inspect Table 6 pages
# Table 6: All Ongoing Projects begins on page 55 and ends on page 156
print("\n--- Scanning Table 6 (Pages 55-156) Sample ---")
p55_text = reader.pages[54].extract_text()
lines_p55 = [line.strip() for line in p55_text.splitlines() if line.strip()]
print(f"Page 55 contains {len(lines_p55)} lines. Sample lines:")
for l in lines_p55[:25]:
    print(" ", l)


PAIMANA PDF Path: ..\data\raw\paimana\FlashReport_March_2026.pdf
File size (MB): 6.55
Total pages in report: 158

--- Scanning Table 6 (Pages 55-156) Sample ---


Page 55 contains 435 lines. Sample lines:
  All Ongoing Projects
  MARCH 2026
  Sl.No
  Project Name
  (Agency)
  (Project Code) (Legacy OCMS Code)
  State
  Date of Approval
  (Start Date)
  MM/YYYY
  Orignal/Target DoC
  (Revised DoC)
  MM/YYYY
  Orignal Cost
  Revised Cost
  in Rs. Crore
  Cumulative
  Expenditure
  in Rs. Crore
  Physical Progress
  (%)
  Ministry of Civil Aviation
  Aviation & Aviation Infrastructure
  1
  Construction of New Domestic Terminal Building Building and miscellaneous works


In [3]:
# Structural summary of Table 6 fields
paimana_profile = {
    "Filename": "FlashReport_March_2026.pdf",
    "Source": "MoSPI / PAIMANA",
    "Total Pages": total_pages,
    "Table 6 Span": "Page 55 to 156 (102 pages)",
    "Estimated Ongoing Projects": 1941,
    "Candidate Key": "Project Code (e.g. 612786) / Legacy OCMS Code",
    "Candidate Target": "Delay Status (revised_doc > original_doc) and delay days",
    "Financial Features": "Original Cost, Revised Cost, Cumulative Expenditure (Rs. Crore)",
    "Progress Feature": "Physical Progress (%)",
    "Timeline Features": "Date of Approval, Start Date, Original DoC, Revised DoC",
    "Geographic Feature": "State (Single State or Multi-States)",
    "Identified Irregularities": "Multiline wrapped parentheses, 6 projects withheld by MoSPI for expenditure checks (ids: 618051, 618307, etc.)"
}
for k, v in paimana_profile.items():
    print(f"{k:25}: {v}")


Filename                 : FlashReport_March_2026.pdf
Source                   : MoSPI / PAIMANA
Total Pages              : 158
Table 6 Span             : Page 55 to 156 (102 pages)
Estimated Ongoing Projects: 1941
Candidate Key            : Project Code (e.g. 612786) / Legacy OCMS Code
Candidate Target         : Delay Status (revised_doc > original_doc) and delay days
Financial Features       : Original Cost, Revised Cost, Cumulative Expenditure (Rs. Crore)
Progress Feature         : Physical Progress (%)
Timeline Features        : Date of Approval, Start Date, Original DoC, Revised DoC
Geographic Feature       : State (Single State or Multi-States)
Identified Irregularities: Multiline wrapped parentheses, 6 projects withheld by MoSPI for expenditure checks (ids: 618051, 618307, etc.)


---
## 2. CENSUS 2011 Dataset (`2011-IndiaStateDistSbDistTwn-0000.xlsx`)
- **Source:** Office of the Registrar General & Census Commissioner, India
- **Purpose:** Baseline demographic, household, literacy, and workforce indicators


In [4]:
census_path = os.path.join(RAW_DIR, "census", "2011-IndiaStateDistSbDistTwn-0000.xlsx")
print("Census Path:", census_path)
print("File size (MB):", round(os.path.getsize(census_path) / (1024 * 1024), 2))

# Load sheet
df_census = pd.read_excel(census_path, sheet_name="Data")
print(f"Dimensions: {df_census.shape[0]} rows, {df_census.shape[1]} columns")
print(f"Exact Duplicate Rows: {df_census.duplicated().sum()}")
print(f"Administrative Levels present: {df_census['Level'].value_counts().to_dict()}")
print(f"TRU Categories: {df_census['TRU'].value_counts().to_dict()}")


Census Path: ..\data\raw\census\2011-IndiaStateDistSbDistTwn-0000.xlsx
File size (MB): 15.3


Dimensions: 28389 rows, 94 columns


Exact Duplicate Rows: 0
Administrative Levels present: {'SUB-DISTRICT': 17964, 'TOWN': 8397, 'DISTRICT': 1920, 'STATE': 105, 'India': 3}
TRU Categories: {'Urban': 15061, 'Total': 6664, 'Rural': 6664}


In [5]:
# Inspect demographic features
demo_cols = ['Level', 'Name', 'TRU', 'No_HH', 'TOT_P', 'TOT_M', 'TOT_F', 'P_LIT', 'TOT_WORK_P', 'NON_WORK_P']
print("Sample District-Level Demographic Records (TRU=Total):")
df_districts = df_census[(df_census['Level'] == 'DISTRICT') & (df_census['TRU'] == 'Total')]
display(df_districts[demo_cols].head(5))

print(f"Total Districts identified: {len(df_districts)}")
print("Missing values in demographic subset:", df_districts[demo_cols].isnull().sum().to_dict())


Sample District-Level Demographic Records (TRU=Total):


,Level,Name,TRU,No_HH,TOT_P,TOT_M,TOT_F,P_LIT,TOT_WORK_P,NON_WORK_P
6,DISTRICT,Kupwara,Total,113929,870354,474190,396164,439654,229064,641290
28,DISTRICT,Badgam,Total,103363,753745,398041,355704,335649,214866,538879
59,DISTRICT,Leh(Ladakh),Total,21909,133487,78971,54516,93770,75079,58408
74,DISTRICT,Kargil,Total,18338,140802,77785,63017,86236,51873,88929
87,DISTRICT,Punch,Total,90261,476835,251899,224936,261724,161393,315442


Total Districts identified: 640
Missing values in demographic subset: {'Level': 0, 'Name': 0, 'TRU': 0, 'No_HH': 0, 'TOT_P': 0, 'TOT_M': 0, 'TOT_F': 0, 'P_LIT': 0, 'TOT_WORK_P': 0, 'NON_WORK_P': 0}


---
## 3. PRIVATE PROPERTY DATA (`ProjectInfo_Gujarat.csv`)
- **Source:** Gujarat Real Estate Regulatory Authority (RERA)
- **Purpose:** Regional private development density, unit capacity, and real-estate market dynamics
- **Boundary Note:** Strictly Gujarat-specific; will not be imputed nationally.


In [6]:
private_path = os.path.join(RAW_DIR, "private_property", "ProjectInfo_Gujarat.csv")
df_private = pd.read_csv(private_path, low_memory=False)
print(f"Dimensions: {df_private.shape[0]} rows, {df_private.shape[1]} columns")
print(f"Exact Duplicate Rows: {df_private.duplicated().sum()}")
print(f"Unique Districts in Gujarat: {df_private['distName'].nunique()}")
print("Top 5 Districts by Project Count:")
print(df_private['distName'].value_counts().head(5))


Dimensions: 14507 rows, 44 columns
Exact Duplicate Rows: 0
Unique Districts in Gujarat: 35
Top 5 Districts by Project Count:
distName
Ahmedabad      4280
Vadodara       2403
Surat          1858
Rajkot         1665
Gandhinagar    1251
Name: count, dtype: int64


In [7]:
# Inspect key numerical and missingness fields
key_private_cols = ['projectRegId', 'projectName', 'projectType', 'distName', 'startDate', 'completionDate', 'totalUnits', 'totalEstimatedCost', 'totalIncurredCost', 'totalAreaOfLand']
print("Missing Value Percentages in Key Columns:")
for c in key_private_cols:
    null_pct = (df_private[c].isnull().sum() / len(df_private)) * 100
    print(f"  - {c:25}: {null_pct:.2f}%")
    
display(df_private[key_private_cols].head(3))


Missing Value Percentages in Key Columns:
  - projectRegId             : 0.00%
  - projectName              : 0.00%
  - projectType              : 0.00%
  - distName                 : 6.03%
  - startDate                : 6.03%
  - completionDate           : 6.03%
  - totalUnits               : 0.14%
  - totalEstimatedCost       : 0.01%
  - totalIncurredCost        : 0.01%
  - totalAreaOfLand          : 39.88%


,projectRegId,projectName,projectType,distName,startDate,completionDate,totalUnits,totalEstimatedCost,totalIncurredCost,totalAreaOfLand
0,7871,SURAT DIAMOND BOURSE,Commercial,Surat,2017-12-05,2023-06-30,4798.0,2.769942e+10,2.769942e+10,143825.4
1,1109,GLOBALE TEXTILE MARKET,Commercial,Surat,2014-07-22,2019-12-31,3754.0,7.110000e+09,7.110000e+09,NaN
2,4258,68 SHOPS + 1228 LIG-2 + 432 MIG-1 AT GOTA,Mixed Development,Ahmedabad,2014-05-13,2018-08-26,3456.0,2.045715e+09,2.045715e+09,NaN


---
## 4. COURT DATA (`Pendency of Court Cases in India.csv`)
- **Source:** India Justice Report / Judicial Statistics
- **Purpose:** State-level judicial capacity, case clearance efficiency, and legal risk indicators


In [8]:
court_path = os.path.join(RAW_DIR, "court", "Pendency of Court Cases in India.csv")
df_court = pd.read_csv(court_path)
print(f"Dimensions: {df_court.shape[0]} rows, {df_court.shape[1]} columns")
print(f"Exact Duplicate Rows: {df_court.duplicated().sum()}")
print("Columns:", df_court.columns.tolist())
display(df_court.head(5))


Dimensions: 37 rows, 8 columns
Exact Duplicate Rows: 0
Columns: ['Unnamed: 0', 'State/UT', 'Budget per capita on judiciary (₹) (2020–21)', 'Population per High Court Judge (2022)', 'Population per Lower Court Judge (2022)', 'Courthall shortfall (%) (2022)', 'Case clearance rate of High Court (2022)', 'Case clearance rate of Lower Court (2022)']


,Unnamed: 0,State/UT,Budget per capita on judiciary (₹) (2020–21),Population per High Court Judge (2022),Population per Lower Court Judge (2022),Courthall shortfall (%) (2022),Case clearance rate of High Court (2022),Case clearance rate of Lower Court (2022)
0,0,India,146,1765760,71224,14.7,95,89.000000
1,1,Andaman and Nicobar Islands (UT),337,1833444,30923,NaN,121,76.000000
2,2,Andhra Pradesh,145,1765733,109673,- 4.0,73,90.000000
3,3,Arunachal Pradesh,199,1681917,44229,36.6,90,91.828571
4,4,Assam,99,1681917,82274,13.4,90,72.000000


In [9]:
# Data type and anomaly inspection
print("Data Types:")
print(df_court.dtypes)
print("\nCourthall Shortfall unique values (notice string negative signs and NaNs):")
print(df_court['Courthall shortfall (%) (2022)'].unique()[:10])


Data Types:
Unnamed: 0                                        int64
State/UT                                            str
Budget per capita on judiciary (₹) (2020–21)      int64
Population per High Court Judge (2022)            int64
Population per Lower Court Judge (2022)           int64
Courthall shortfall (%) (2022)                      str
Case clearance rate of High Court (2022)          int64
Case clearance rate of Lower Court (2022)       float64
dtype: object

Courthall Shortfall unique values (notice string negative signs and NaNs):
<ArrowStringArray>
['14.7', nan, '- 4.0', '36.6', '13.4', '20.2', '- 3.3', '2.7', '- 14.3', '32.5']
Length: 10, dtype: str


---
## 5. POWER INFRASTRUCTURE (`India_Statewise_Power_Infrastructure_Data_RBI.csv`)
- **Source:** Reserve Bank of India (RBI) Database on Indian Economy
- **Purpose:** State-level power availability, demand, and grid infrastructure adequacy


In [10]:
power_path = os.path.join(RAW_DIR, "power", "India_Statewise_Power_Infrastructure_Data_RBI.csv")
df_power = pd.read_csv(power_path)
print(f"Dimensions: {df_power.shape[0]} rows, {df_power.shape[1]} columns")
print(f"Exact Duplicate Rows: {df_power.duplicated().sum()}")
print("Fiscal Years Available:", df_power['Year'].unique().tolist())
print("Unique States/UTs:", df_power['State/Union Territory'].nunique())
display(df_power.head(5))


Dimensions: 612 rows, 6 columns
Exact Duplicate Rows: 0
Fiscal Years Available: ['2004-05', '2005-06', '2006-07', '2007-08', '2008-09', '2009-10', '2010-11', '2011-12', '2012-13', '2013-14', '2014-15', '2015-16', '2016-17', '2017-18', '2018-19', '2019-20', '2020-21']
Unique States/UTs: 36


,State/Union Territory,Year,Power_Requirement_Net_Crore_Units,Availability_Of_Power_Net_Crore_Units,Availability_Of_Power_Per_Capita_kiloWatt-Hour,Installed_Power_Capacity_MegaWatt
0,Andaman and Nicobar Islands,2004-05,-,-,-,65
1,Andhra Pradesh,2004-05,5042,5006,656.9,10809
2,Arunachal Pradesh,2004-05,16,16,143.9,187
3,Assam,2004-05,379,358,134.4,1133
4,Bihar,2004-05,720,648,78,1644


In [11]:
# Inspect placeholder '-' strings and data conversion needs
print("Count of '-' placeholders per column:")
for col in df_power.columns:
    dash_count = (df_power[col] == '-').sum()
    print(f"  - {col:45}: {dash_count} instances")


Count of '-' placeholders per column:
  - State/Union Territory                        : 0 instances
  - Year                                         : 0 instances
  - Power_Requirement_Net_Crore_Units            : 3 instances
  - Availability_Of_Power_Net_Crore_Units        : 3 instances
  - Availability_Of_Power_Per_Capita_kiloWatt-Hour: 3 instances
  - Installed_Power_Capacity_MegaWatt            : 0 instances


---
## 6. ROAD TRANSPORT INFRASTRUCTURE (BRS 2018-19 Annexures)
- **Source:** Ministry of Road Transport and Highways (MoRTH) - Basic Road Statistics
- **Purpose:** Regional physical connectivity, highway network density, and transport logistics capacity


In [12]:
roads_dir = os.path.join(RAW_DIR, "roads")
ann3_1 = pd.read_csv(os.path.join(roads_dir, "Road_Transport_BRS_2018-19_Annexure3_1.csv"))
ann7_9b = pd.read_csv(os.path.join(roads_dir, "Road_Transport_BRS_2018-19_Annexure7_9b.csv"))
ann7_9c = pd.read_csv(os.path.join(roads_dir, "Road_Transport_BRS_2018-19_Annexure7_9c.csv"))

print(f"Annexure 3.1 (Surfaced Roads)     : {ann3_1.shape[0]} rows, {ann3_1.shape[1]} cols")
print(f"Annexure 7.9b (Density / 1000 sq km): {ann7_9b.shape[0]} rows, {ann7_9b.shape[1]} cols")
print(f"Annexure 7.9c (Length / 1000 pop)  : {ann7_9c.shape[0]} rows, {ann7_9c.shape[1]} cols")

print("\nSample State Names with Formatting Noise (trailing spaces, *, $):")
print("  Ann 3.1 :", ann3_1['Name of State / UT'].dropna().tolist()[:3])
print("  Ann 7.9b:", ann7_9b['Name of the States'].dropna().tolist()[:3])
print("  Ann 7.9c:", ann7_9c['Name of the States'].dropna().tolist()[:3])


Annexure 3.1 (Surfaced Roads)     : 37 rows, 4 cols
Annexure 7.9b (Density / 1000 sq km): 36 rows, 16 cols
Annexure 7.9c (Length / 1000 pop)  : 36 rows, 16 cols

Sample State Names with Formatting Noise (trailing spaces, *, $):
  Ann 3.1 : ['Andhra Pradesh ', 'Arunachal Pradesh $', 'Assam']
  Ann 7.9b: [' Andhra Pradesh(*)', ' Arunachal Pradesh', ' Assam']
  Ann 7.9c: [' Andhra Pradesh(*)', ' Arunachal Pradesh', ' Assam']


---
## 7. Cross-Dataset Harmonization & Key Alignment Matrix

To prevent artificial row inflation, the master dataset must maintain **PAIMANA project-level granularity**, attaching secondary datasets via aggregated spatial keys (`state` and/or `district`).


In [13]:
alignment_summary = pd.DataFrame([
    {"Dataset": "PAIMANA Projects", "Granularity": "Project level (~1,941)", "Primary Key": "project_code", "Join Key to Master": "Target Backbone"},
    {"Dataset": "Census 2011", "Granularity": "District / State level", "Primary Key": "State / District code", "Join Key to Master": "state, district"},
    {"Dataset": "Gujarat RERA", "Granularity": "District aggregated", "Primary Key": "distName", "Join Key to Master": "state='Gujarat', district"},
    {"Dataset": "Court Pendency", "Granularity": "State level (36 States/UTs)", "Primary Key": "State/UT", "Join Key to Master": "state"},
    {"Dataset": "Power Infra", "Granularity": "State level (latest 2020-21)", "Primary Key": "State/Union Territory", "Join Key to Master": "state"},
    {"Dataset": "Road Transport", "Granularity": "State level (36 States/UTs)", "Primary Key": "Name of State", "Join Key to Master": "state"}
])
display(alignment_summary)


,Dataset,Granularity,Primary Key,Join Key to Master
0,PAIMANA Projects,"Project level (~1,941)",project_code,Target Backbone
1,Census 2011,District / State level,State / District code,"state, district"
2,Gujarat RERA,District aggregated,distName,"state='Gujarat', district"
3,Court Pendency,State level (36 States/UTs),State/UT,state
4,Power Infra,State level (latest 2020-21),State/Union Territory,state
5,Road Transport,State level (36 States/UTs),Name of State,state


---
## 8. Summary of Findings & Next Pipeline Steps

1. **PAIMANA Extraction:** High-fidelity regex extraction across Table 6 (pages 55-156) will be executed into `data/processed/paimana/paimana_projects.csv`.
2. **State Name Standardization:** A unified dictionary mapping state names across Census, RBI Power, Courts, and MoRTH will be built in `src/preprocessing/`.
3. **Data Quality Governance:** 
   - No data fabrication for missing cells.
   - Separate interim and processed outputs.
   - Clean verification before model training.
